# PSN-2 Training — Kaggle
Upload `psn2_kaggle_full.tar.gz` as a dataset attachment before running.

In [ ]:
import os, subprocess, sys

# ── Locate the archive (dataset attachment or working dir) ──────────────────
ARCHIVE = None
ARCHIVE_TYPE = None
search_paths = ['/kaggle/input', '/kaggle/working', '.']
for root in search_paths:
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f in ('psn2_kaggle_full.zip', 'psn2_kaggle_full.tar.gz'):
                ARCHIVE = os.path.join(dirpath, f)
                ARCHIVE_TYPE = 'zip' if f.endswith('.zip') else 'tar'
                break
        if ARCHIVE:
            break
    if ARCHIVE:
        break

assert ARCHIVE, 'psn2_kaggle_full.zip not found — attach it as a dataset'
print('Found archive:', ARCHIVE, '| type:', ARCHIVE_TYPE)

In [ ]:
import zipfile, tarfile
WORKDIR = '/kaggle/working/psn2_repo'
os.makedirs(WORKDIR, exist_ok=True)
if ARCHIVE_TYPE == 'zip':
    with zipfile.ZipFile(ARCHIVE) as zf:
        zf.extractall(WORKDIR)
else:
    with tarfile.open(ARCHIVE) as tf:
        tf.extractall(WORKDIR)
sys.path.insert(0, WORKDIR)
os.chdir(WORKDIR)
print('Extracted to', WORKDIR)
print(os.listdir(WORKDIR))

In [ ]:
# Install any missing deps (torch is pre-installed on Kaggle)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm'], check=True)

In [ ]:
import json, torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Config override for Kaggle constraints ───────────────────────────────────
cfg_path = os.path.join(WORKDIR, 'configs/default.json')
with open(cfg_path) as f:
    cfg = json.load(f)

cfg.update({
    'vsa_dim':        512,
    'max_nodes':      256,
    'grid_size':      8,
    'grid_vocab':     10,
    'rel_vocab_size': 64,
    'batch_size':     32,
    'steps':          10000,
    'lr':             1e-3,
    'log_every':      100,
    'checkpoint_every': 500,
    'checkpoint_dir': '/kaggle/working/artifacts',
    'stage':          'D1',
    'train_mix':      {'arc': 0.6, 'graph': 0.4},
})

with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)

print('Config saved:', cfg)

In [ ]:
# ── Training ─────────────────────────────────────────────────────────────────
# Check for existing checkpoint to resume
latest_ckpt = '/kaggle/working/artifacts/latest.pt'
resume_flag = ['--resume', latest_ckpt] if os.path.exists(latest_ckpt) else []

cmd = [
    sys.executable, 'train.py',
    '--config', cfg_path,
] + resume_flag

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=WORKDIR)
print('Exit code:', result.returncode)

In [ ]:
# ── Evaluation ───────────────────────────────────────────────────────────────
ckpt_path = '/kaggle/working/artifacts/latest.pt'
if os.path.exists(ckpt_path):
    result = subprocess.run([
        sys.executable, 'evaluate.py',
        '--config', cfg_path,
        '--checkpoint', ckpt_path,
        '--output', '/kaggle/working/eval_results.json',
    ], cwd=WORKDIR)
    print('Eval exit code:', result.returncode)
else:
    print('No checkpoint found — run training first')

In [ ]:
# ── List output artifacts ─────────────────────────────────────────────────────
artifacts_dir = '/kaggle/working/artifacts'
if os.path.exists(artifacts_dir):
    for f in sorted(os.listdir(artifacts_dir)):
        size_mb = os.path.getsize(os.path.join(artifacts_dir, f)) / 1e6
        print(f'  {f:40s}  {size_mb:.1f} MB')
else:
    print('No artifacts yet')